# 데이터 전처리 — 중복 제거

수집·병합한 통합본(`kpop_radar_2023_2025_dedup_merged_features.csv`)에서 **같은 곡의 중복 영상을 정리**해 분석용 데이터(`data(drop_duplicated).csv`)를 만드는 노트북입니다. 데이터를 확인하면서 중복을 어떻게 찾았고 어떤 기준으로 정리했는지 **과정을 단계별로** 남겼습니다.

같은 곡이 여러 영상으로 올라와 있으면(원곡 공식 채널, 음원 유통사·음원 플랫폼 채널 등) 분석에서 같은 곡이 여러 번 집계됩니다. 이 노트북은 행을 골라낼 뿐 **값이나 컬럼은 바꾸지 않습니다**(출력의 모든 행은 입력의 행 그대로입니다).

## 진행 순서

| 섹션 | 내용 |
|---|---|
| **0. 데이터 확인** | 통합본을 불러와 크기, 컬럼, 장르 라벨, 중복 정도를 확인 |
| **1. 중복 후보 분리** | 곡명+아티스트가 같은 행(후보)과 그렇지 않은 행으로 나눔 |
| **2. video_id 중복 제거** | 후보 중 같은 영상(`video_id`)을 하나로 정리 |
| **3. 채널 중복 확인** | 같은 곡이 몇 개 채널에 올라와 있는지, 어떤 채널이 많은지 확인 |
| **4. 유통사 채널 제거** | 유통사 채널 목록을 1차 → 최종으로 넓혀 가며 제거, 남은 중복은 수동으로 정리 |
| **5. 병합·검증·비교** | 비후보와 합쳐 검증하고, 기존 결과와 비교 (`SAVE_OUTPUT = True`일 때만 저장) |

**입력:** 수집한 특징(시각·오디오·가사/장르)에 가사 통계·감성·토픽·아티스트 정보가 더해진 통합본 → **출력:** `data(drop_duplicated).csv` (기본 설정에서는 저장하지 않고 기존 파일과 비교만 합니다)

**참고사항**
- 이 통합본에서는 2,545행이 **2,043행**이 되며, 결과는 기존 `data(drop_duplicated).csv`와 같습니다. 출력 파일이 이미 있으면 새 결과와 자동으로 비교하며, 기본 설정(`SAVE_OUTPUT = False`)에서는 **파일을 저장하지 않고 비교 결과만** 출력합니다.
- 후보 표시(1번)를 `video_id` 중복 제거(2번)보다 **먼저** 합니다. 그래서 같은 영상이 여러 행으로 중복돼 있던 곡이 유통사 채널에만 올라와 있으면 4번에서 **곡 자체가 제외**됩니다(이 통합본에서는 13곡). 이 순서가 기존 결과를 재현하는 방식이라 그대로 두었습니다.
- 확인용 출력에는 가사 원문 등 긴 텍스트 컬럼을 넣지 않고 곡명·채널·조회수 등만 보여줍니다(`VIEW_COLS`).


## 설정

입출력 파일, 저장 여부, 확인용 출력 컬럼을 정의합니다. **가장 먼저 실행하세요.**

- `INPUT_FILE` / `OUTPUT_FILE` — 통합본과 결과 파일 (`data_preprocessing/` 기준 `../data/`)
- `SAVE_OUTPUT` — 결과를 `OUTPUT_FILE`로 저장할지 여부. 기본값 `False`는 **저장하지 않고 기존 파일과 비교만** 합니다(기존 파일은 그대로 유지). `True`로 바꾸면 저장합니다.
- `VIEW_COLS` — 중복 영상을 확인할 때 보여줄 컬럼

In [1]:
# 공통 설정: 입출력 파일, 저장 여부, 확인용 출력 컬럼
from pathlib import Path

DATA_DIR = Path("../data")                                                    # 통합본과 결과 파일이 있는 폴더
INPUT_FILE = DATA_DIR / "kpop_radar_2023_2025_dedup_merged_features.csv"      # 통합본
OUTPUT_FILE = DATA_DIR / "data(drop_duplicated).csv"                          # 중복 제거 결과
SAVE_OUTPUT = False   # True로 바꾸면 결과를 OUTPUT_FILE로 저장 (기본: 저장하지 않고 기존 파일과 비교만)

# 확인용 출력에서 보여줄 컬럼 (가사 원문 등 긴 텍스트 컬럼은 출력하지 않음)
VIEW_COLS = ['songName', 'artists', 'api_channel_title', 'api_view_count', 'api_published_at', 'video_id', 'url']

# 0. 데이터 확인
<small>통합본을 불러와 크기, 컬럼, 장르 라벨, 중복 정도를 확인합니다. 이 통합본은 인덱스 컬럼(`Unnamed`)이 이미 없고 장르 라벨도 통일돼 있어서(`POP / 락` 없음) 아래 정리 단계는 결과를 바꾸지 않습니다. 정리 과정을 남기기 위해 그대로 두었습니다.</small>

In [2]:
# 0-1. 통합본 불러오기
import pandas as pd

final_df = pd.read_csv(INPUT_FILE, encoding='utf-8-sig')
print(f"{final_df.shape[0]:,}행 × {final_df.shape[1]}컬럼")

2,545행 × 59컬럼


In [3]:
final_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2545 entries, 0 to 2544
Data columns (total 59 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   url                     2545 non-null   str    
 1   songName                2545 non-null   str    
 2   artists                 2545 non-null   str    
 3   video_id                2545 non-null   str    
 4   api_view_count          2545 non-null   float64
 5   api_like_count          2545 non-null   float64
 6   api_comment_count       2545 non-null   float64
 7   api_published_at        2545 non-null   str    
 8   api_duration            2545 non-null   str    
 9   api_channel_title       2545 non-null   str    
 10  valence_raw             2545 non-null   float64
 11  arousal_raw             2545 non-null   float64
 12  valence_normalized      2545 non-null   float64
 13  arousal_normalized      2545 non-null   float64
 14  energy                  2545 non-null   float64
 15

In [4]:
final_df[VIEW_COLS].head()

,songName,artists,api_channel_title,api_view_count,api_published_at,video_id,url
0,APT. (ROSÉ & Bruno Mars),684|로제 (ROSÉ)|ROSE|1,ROSÉ,2.309341e+09,2024-10-18 04:00:07+00:00,ekr2nIex040,https://www.youtube.com/watch?v=ekr2nIex040
1,DRIP,2543|BABYMONSTER (베이비몬스터)|BABYMONSTER|1,BABYMONSTER,3.421201e+08,2024-11-01 04:00:06+00:00,Zp-Jhuhq0bQ,https://www.youtube.com/watch?v=Zp-Jhuhq0bQ
2,Strategy (feat. Megan Thee Stallion),5|TWICE (트와이스)|TWICE|1,JYP Entertainment,1.377780e+08,2024-12-06 04:59:07+00:00,Sz_wWzgh-vQ,https://www.youtube.com/watch?v=Sz_wWzgh-vQ
3,toxic till the end,684|로제 (ROSÉ)|ROSE|1,ROSÉ,1.262376e+08,2024-12-06 05:00:06+00:00,eA0lHNZ1KCA,https://www.youtube.com/watch?v=eA0lHNZ1KCA
4,Love In My Heart,2543|BABYMONSTER (베이비몬스터)|BABYMONSTER|1,BABYMONSTER,7.952422e+07,2024-12-15 15:00:07+00:00,1kXLsrun51s,https://www.youtube.com/watch?v=1kXLsrun51s


In [5]:
final_df[VIEW_COLS].tail()

,songName,artists,api_channel_title,api_view_count,api_published_at,video_id,url
2540,Rover,397|카이 (KAI)|KAI|1,SMTOWN,100537798.0,2023-03-13 08:59:10+00:00,zlTIextYnyQ,https://www.youtube.com/watch?v=zlTIextYnyQ
2541,늦은 말 (Promise),527|도영 (DOYOUNG)|DOYOUNG|1,SMTOWN,470318.0,2025-12-09 09:01:04+00:00,zpPFHclAOoM,https://www.youtube.com/watch?v=zpPFHclAOoM
2542,EXTRA,1332|전소미|JEONSOMI|1,THEBLACKLABEL,16010665.0,2025-07-07 08:59:59+00:00,zq5mr3ePY1I,https://www.youtube.com/watch?v=zq5mr3ePY1I
2543,ME+YOU,5|TWICE (트와이스)|TWICE|1,JYP Entertainment,9015438.0,2025-10-10 03:58:07+00:00,zqorlX_5oHQ,https://www.youtube.com/watch?v=zqorlX_5oHQ
2544,Inside My Love,2688|RIIZE (라이즈)|RIIZE|1,RIIZE,1195354.0,2025-05-19 09:05:00+00:00,ztoPhcfnj3Q,https://www.youtube.com/watch?v=ztoPhcfnj3Q


In [6]:
# 인덱스 컬럼(Unnamed) 확인 및 제거
unnamed = [c for c in final_df.columns if c.startswith('Unnamed')]
final_df = final_df.drop(columns=unnamed)
print(f"제거한 인덱스 컬럼: {unnamed}")

제거한 인덱스 컬럼: []


In [7]:
final_df['genie_genre'].unique()

<StringArray>
['락', '댄스', '랩/힙합', '발라드', '팝', '일렉트로니카', 'R&B/소울', '인디']
Length: 8, dtype: str

In [8]:
# 장르 라벨 통일 (POP / 락 → 락)
final_df['genie_genre'] = final_df['genie_genre'].replace({
    'POP / 락': '락'
})
final_df['genie_genre'].value_counts()

genie_genre
댄스        1144
발라드        434
락          333
R&B/소울     264
랩/힙합       227
인디          75
일렉트로니카      43
팝           25
Name: count, dtype: int64

In [9]:
# 개수 확인: 행 수, 고유 video_id 수, 고유 곡 수
n_video = final_df['video_id'].nunique()
print(f"전체 행: {len(final_df):,}")
print(f"고유 video_id: {n_video:,}  (같은 영상이 여러 행으로 들어 있는 경우: {len(final_df) - n_video:,}행)")
print(f"고유 곡(songName+artists): {final_df.drop_duplicates(['songName', 'artists']).shape[0]:,}")

전체 행: 2,545
고유 video_id: 2,297  (같은 영상이 여러 행으로 들어 있는 경우: 248행)
고유 곡(songName+artists): 2,053


> **확인한 것:** 2,545행 중 고유 `video_id`는 2,297개입니다. 같은 영상이 여러 행으로 들어 있고(248행), 곡 기준으로는 2,053곡입니다. 곡 하나에 서로 다른 영상이 여러 개 있는 경우도 많아서(2,297개 영상 → 2,053곡) 중복을 두 단계(같은 영상 → 같은 곡)로 나눠 정리합니다.

# 1. 중복 처리
## 1. songName+artists 기준 중복 후보 추출
<small>곡명+아티스트가 같은 행을 모두 찾아 `df_dup`(중복 후보)와 `df_unique`(애초에 중복 없는 곡)로 나눕니다. 이 통합본에서는 후보 912행, 비후보 1,633행입니다. 후보 표시는 이 단계에서 **행 단위**로 먼저 하고, `video_id` 중복 제거는 그 다음에 합니다.</small>

In [10]:
# 1. 중복된 것들만 추출 (모든 행 포함)
dup_mask = final_df.duplicated(subset=['songName', 'artists'], keep=False)

In [11]:
dup_mask.sum()

np.int64(912)

In [12]:
df_dup = final_df[dup_mask].reset_index(drop=True)      # 중복 처리 필요한 것들
df_unique = final_df[~dup_mask].reset_index(drop=True)  # 처음부터 중복 없는 것들

print(f"원본: {len(final_df)} rows")
print(f"중복 없는 것 (df_unique): {len(df_unique)} rows")
print(f"중복 있는 것 (df_dup): {len(df_dup)} rows")
print(f"합계 확인: {len(df_unique) + len(df_dup)} rows")

원본: 2545 rows
중복 없는 것 (df_unique): 1633 rows
중복 있는 것 (df_dup): 912 rows
합계 확인: 2545 rows


## 2. video_id 기준 1차 중복 제거
<small>중복 후보 중 `video_id`가 같은 행(완전히 동일한 영상)을 먼저 하나만 남깁니다. 이 통합본에서는 912행 → 664행으로 248행이 줄어듭니다.</small>

In [13]:
df_dup_deduped = (
    df_dup
    .drop_duplicates(subset=['video_id'], keep='first')
    .reset_index(drop=True)
)

print(f"중복 제거 전: {len(df_dup)} rows")
print(f"중복 제거 후: {len(df_dup_deduped)} rows")
print(f"제거된 행: {len(df_dup) - len(df_dup_deduped)} rows")

중복 제거 전: 912 rows
중복 제거 후: 664 rows
제거된 행: 248 rows


In [14]:
# 확인: video_id가 행마다 유일해졌는가
print(f"video_id 고유 수: {df_dup_deduped['video_id'].nunique()}")
print(f"행 수: {len(df_dup_deduped)}")

video_id 고유 수: 664
행 수: 664


## 3. 유통사 채널 중복 확인
<small>같은 곡이 몇 개의 서로 다른 채널(`api_channel_title`)에 걸쳐 남아있는지 집계합니다. `video_id`를 정리한 뒤에도 같은 곡이 여러 채널에 올라와 있는 곡이 237곡 있고, 그 안에서 많이 나오는 채널은 1theK (원더케이) 138행, Stone Music Entertainment 23행, Dreamus Music 17행 순입니다. 원곡 공식 채널이 아니라 음원 유통사·음원 플랫폼의 채널이 반복해서 나타납니다.</small>

In [15]:
result = df_dup_deduped.groupby(['songName', 'artists'])['api_channel_title'].nunique()
print(f"채널이 2개 이상인 곡: {len(result[result >= 2])}개")

채널이 2개 이상인 곡: 237개


In [16]:
# 실제로 어떤 채널들이 있는지 확인 (앞 10곡)
multi_channel_songs = result[result >= 2].index
df_dup_deduped[df_dup_deduped.set_index(['songName', 'artists']).index.isin(multi_channel_songs)]\
    .groupby(['songName', 'artists'])['api_channel_title'].apply(list).head(10)

songName      artists                            
2 Months      3612|유아유 (UAU)|UAU|0                    [1theK (원더케이), Dreamcatcher official]
ATE THAT      2756|YOUNG POSSE (영파씨)|YOUNGPOSSE|1         [YOUNG POSSE • 영파씨, 1theK (원더케이)]
Ain't Nobody  2932|VVUP(비비업)|VVUP|1                                     [VVUP, GENIE MUSIC]
BANG OUT      2772|WHIB(휘브)|WHIB|0                                     [WHIB, 1theK (원더케이)]
BEBE          2031|STAYC(스테이씨)|STAYC|1                                [1theK (원더케이), STAYC]
BUBBLE GUM    2242|Kep1er|Kep1er|1                      [Kep1er, Stone Music Entertainment]
Blessed       581|하성운|HA_SUNG_WOON|1                      [BPM Entertainment, 1theK (원더케이)]
Bloom         1083|볼빨간사춘기|BOL4|1                             [SUPER SOUND Bugs!, 쇼파르엔터테인먼트]
Bora          718|이민혁 (HUTA)|HUTA|1                  [1theK (원더케이), 비투비 컴퍼니 (BTOB COMPANY)]
Broken Party  402|첸 (CHEN)|CHEN|1                    [INB100, 워너뮤직코리아 (Warner Music Korea)]
Name: api_channel_title, dtype

In [17]:
# 어떤 채널이 많이 나오는지 확인 (상위 10개)
df_multi = df_dup_deduped[
    df_dup_deduped.set_index(['songName', 'artists']).index.isin(multi_channel_songs)
]

# 중복 채널로 많이 등장하는 채널명 순위
df_multi['api_channel_title'].value_counts().head(10)

api_channel_title
1theK (원더케이)                    138
Stone Music Entertainment        23
Dreamus Music                    17
워너뮤직코리아 (Warner Music Korea)     14
GENIE MUSIC                      12
SUPER SOUND Bugs!                10
월간 윤종신                            8
MAMAMOO                           5
비투비 컴퍼니 (BTOB COMPANY)            5
ONEUS                             5
Name: count, dtype: int64

## 4. 유통사 채널 제거 (REMOVE_LIST 1차 적용)
<small>1theK, Stone Music Entertainment, Dreamus Music 등 알려진 유통사 공식 업로드 채널을 제거해 곡당 채널 수를 줄입니다. 먼저 1차 목록(5개 채널)으로 시작합니다.</small>

In [18]:
REMOVE_LIST_1 = ['1theK (원더케이)', 'Stone Music Entertainment', 'Dreamus Music',
                 '워너뮤직코리아 (Warner Music Korea)', 'GENIE MUSIC']

# 각 곡별로 유통사 채널이 몇 개인지 세는 함수
def count_dist_channels(df, remove_list):
    return df.groupby(['songName', 'artists'])['api_channel_title'].apply(lambda s: s.isin(remove_list).sum())

dist_count = count_dist_channels(df_dup_deduped, REMOVE_LIST_1)
print(f"유통사 채널이 2개 이상인 곡: {(dist_count >= 2).sum()}개")
dist_count[dist_count >= 2].head(10)

유통사 채널이 2개 이상인 곡: 1개


songName           artists   
교회오빠 (Feat.BOBBY)  1564|오반||0    2
Name: api_channel_title, dtype: int64

> 유통사 채널이 2개 이상인 곡은 1곡(`교회오빠 (Feat.BOBBY)`)입니다. 직접 열어 어떤 영상들이 있는지 확인합니다.

In [19]:
df_dup_deduped.loc[df_dup_deduped['songName'] == '교회오빠 (Feat.BOBBY)', VIEW_COLS]

,songName,artists,api_channel_title,api_view_count,api_published_at,video_id,url
612,교회오빠 (Feat.BOBBY),1564|오반||0,워너뮤직코리아 (Warner Music Korea),2436.0,2025-03-05 09:00:57+00:00,my-Mrod6cJU,https://www.youtube.com/watch?v=my-Mrod6cJU
621,교회오빠 (Feat.BOBBY),1564|오반||0,1theK (원더케이),55464.0,2025-03-13 09:00:58+00:00,oHyzyV1qOVY,https://www.youtube.com/watch?v=oHyzyV1qOVY


In [20]:
df_dup_deduped.loc[df_dup_deduped['url'] == 'https://www.youtube.com/watch?v=my-Mrod6cJU', VIEW_COLS]

,songName,artists,api_channel_title,api_view_count,api_published_at,video_id,url
612,교회오빠 (Feat.BOBBY),1564|오반||0,워너뮤직코리아 (Warner Music Korea),2436.0,2025-03-05 09:00:57+00:00,my-Mrod6cJU,https://www.youtube.com/watch?v=my-Mrod6cJU


> 이 곡은 워너뮤직코리아가 올린 15초짜리 짧은 영상과 1theK의 영상이 함께 있어 유통사 채널이 2개입니다. 짧은 영상을 먼저 제거하고 다시 확인하면 유통사 채널이 2개 이상인 곡이 0곡이 됩니다.

In [21]:
# url 기준으로 해당 행의 index 찾아서 drop
idx = df_dup_deduped[df_dup_deduped['url'] == 'https://www.youtube.com/watch?v=my-Mrod6cJU'].index
print(idx)

df_dup_deduped = df_dup_deduped.drop(idx).reset_index(drop=True)
print(f"제거 후: {len(df_dup_deduped)} rows")

RangeIndex(start=612, stop=613, step=1)
제거 후: 663 rows


In [22]:
df_dup_deduped.loc[df_dup_deduped['url'] == 'https://www.youtube.com/watch?v=my-Mrod6cJU', VIEW_COLS]

,songName,artists,api_channel_title,api_view_count,api_published_at,video_id,url


In [23]:
# 재확인: 유통사 채널이 2개 이상인 곡
dist_count = count_dist_channels(df_dup_deduped, REMOVE_LIST_1)
print(f"유통사 채널이 2개 이상인 곡: {(dist_count >= 2).sum()}개")
dist_count[dist_count >= 2].head(10)

유통사 채널이 2개 이상인 곡: 0개


Series([], Name: api_channel_title, dtype: int64)

### 4-1. 1차 목록(5개 채널) 적용
<small>1차 목록의 채널이 올린 영상을 제거하고, 그래도 채널이 2개 이상 남은 곡을 다시 확인합니다. 이 통합본에서는 212행이 제거되고, 채널이 2개 이상 남은 곡은 36곡이며 남은 중복 채널은 SUPER SOUND Bugs! 10행, DanalEntertainment 5행, SEOUL MUSIC / 서울뮤직 3행 순입니다. 이 목록 밖에도 유통사 채널이 더 있다는 뜻입니다.</small>

In [24]:
df_dup_v5 = df_dup_deduped[
    ~df_dup_deduped['api_channel_title'].isin(REMOVE_LIST_1)
]

print(f"제거 전: {len(df_dup_deduped)} rows")
print(f"제거 후: {len(df_dup_v5)} rows")
print(f"제거된 행: {len(df_dup_deduped) - len(df_dup_v5)} rows")

result_v5 = df_dup_v5.groupby(['songName', 'artists'])['api_channel_title'].nunique()
print(f"\n아직 채널 2개 이상인 곡: {len(result_v5[result_v5 >= 2])}개")

# 남은 중복에서 어떤 채널이 많은지 확인 (상위 10개)
multi_channel_songs_v5 = result_v5[result_v5 >= 2].index
df_multi_v5 = df_dup_v5[
    df_dup_v5.set_index(['songName', 'artists']).index.isin(multi_channel_songs_v5)
]
print("\n남은 중복 채널 순위:")
print(df_multi_v5['api_channel_title'].value_counts().head(10))

제거 전: 663 rows
제거 후: 451 rows
제거된 행: 212 rows

아직 채널 2개 이상인 곡: 36개

남은 중복 채널 순위:
api_channel_title
SUPER SOUND Bugs!      10
DanalEntertainment      5
SEOUL MUSIC / 서울뮤직      3
쇼파르엔터테인먼트               3
온리원 뮤비                  3
Blackswan Official      2
Brave Entertainment     2
TIOT 티아이오티              2
AIMERS                  2
MUSIC&NEW 뮤직앤뉴          2
Name: count, dtype: int64


### 4-2. 유통사 채널 제거 최종 리스트 적용
<small>남은 중복에 나타난 유통사 채널들을 추가해 목록을 최종(21개 채널)으로 확장한 뒤 다시 필터링합니다. 최종 목록으로 다시 세어도 한 곡에 유통사 채널이 2개 이상 겹치는 경우는 0곡이라, 목록을 이 정도로 넓히면 충분합니다.</small>

In [25]:
REMOVE_LIST = ['1theK (원더케이)', 'Stone Music Entertainment', 'Dreamus Music',
               '워너뮤직코리아 (Warner Music Korea)', 'GENIE MUSIC',
               'SUPER SOUND Bugs!', 'DanalEntertainment', '소니뮤직코리아 Sony Music Korea',
               'Sound Republica', 'MUSIC&NEW 뮤직앤뉴', 'SEOUL MUSIC / 서울뮤직',
               'YOU Entertainment', 'YY Entertainment', 'AT AREA',
               'VLENDING MUSIC', '온리원 뮤비',
               'ZENITH CNM', 'TM ENTERTAINMENT', 'OGAM Entertainment',
               'Studio M-Lab', '더 볼트 THE VAULT']

# 각 곡별로 유통사 채널이 몇 개인지 확인
dist_count = count_dist_channels(df_multi_v5, REMOVE_LIST)
print(f"유통사 채널이 2개 이상인 곡: {(dist_count >= 2).sum()}개")
dist_count[dist_count >= 2].head(10)

유통사 채널이 2개 이상인 곡: 0개


Series([], Name: api_channel_title, dtype: int64)

In [26]:
df_dup_v6 = df_dup_deduped[
    ~df_dup_deduped['api_channel_title'].isin(REMOVE_LIST)
]

print(f"제거 전: {len(df_dup_deduped)} rows")
print(f"제거 후: {len(df_dup_v6)} rows")
print(f"제거된 행: {len(df_dup_deduped) - len(df_dup_v6)} rows")

result_v6 = df_dup_v6.groupby(['songName', 'artists'])['api_channel_title'].nunique()
print(f"\n아직 채널 2개 이상인 곡: {len(result_v6[result_v6 >= 2])}개")

제거 전: 663 rows
제거 후: 414 rows
제거된 행: 249 rows

아직 채널 2개 이상인 곡: 2개


In [27]:
# 아직 채널이 2개 이상 남은 곡 (앞 15곡)
multi_v6 = result_v6[result_v6 >= 2].index
df_dup_v6[df_dup_v6.set_index(['songName', 'artists']).index.isin(multi_v6)]\
    .groupby(['songName', 'artists'])['api_channel_title'].apply(list).head(15)

songName      artists    
MOVIE         2921|박제업||0    [그린유니버스뮤직 Green Universe Music (GU Music), PAR...
영화 한편 본 것 같아  1561|송하예||0           [에잇 8recordz x studio8, 플랩 [Playlist Lab]]
Name: api_channel_title, dtype: object

### 4-3. 남은 중복 수동 정리
<small>최종 목록으로 제거한 뒤에도 같은 곡이 여러 채널에 남은 곡이 2곡(`MOVIE`, `영화 한편 본 것 같아`) 있습니다. 이 두 곡과, 유통사가 아닌 채널의 영상이라 자동 규칙으로 걸러지지 않은 `Let Me Leave You`를 직접 확인해서 제외합니다.</small>

In [28]:
df_multi_v5.loc[df_multi_v5['songName'] == '영화 한편 본 것 같아', VIEW_COLS]

,songName,artists,api_channel_title,api_view_count,api_published_at,video_id,url
389,영화 한편 본 것 같아,1561|송하예||0,에잇 8recordz x studio8,1851.0,2025-09-17 09:00:23+00:00,Fag8mxHphxA,https://www.youtube.com/watch?v=Fag8mxHphxA
578,영화 한편 본 것 같아,1561|송하예||0,플랩 [Playlist Lab],123.0,2025-09-17 09:01:12+00:00,i2FkyWgBMA4,https://www.youtube.com/watch?v=i2FkyWgBMA4


In [29]:
df_dup_v6.loc[df_dup_v6['songName'] == '영화 한편 본 것 같아', VIEW_COLS]

,songName,artists,api_channel_title,api_view_count,api_published_at,video_id,url
389,영화 한편 본 것 같아,1561|송하예||0,에잇 8recordz x studio8,1851.0,2025-09-17 09:00:23+00:00,Fag8mxHphxA,https://www.youtube.com/watch?v=Fag8mxHphxA
578,영화 한편 본 것 같아,1561|송하예||0,플랩 [Playlist Lab],123.0,2025-09-17 09:01:12+00:00,i2FkyWgBMA4,https://www.youtube.com/watch?v=i2FkyWgBMA4


> 같은 곡의 중복 영상이지만 유통사명 또는 video_id가 기존 항목과 겹쳐서 자동 필터(유통사 채널 제거, video_id 중복 제거)에 걸러지지 않고 남아있던 케이스라 수동으로 제거합니다.

In [30]:
df_dup_v6 = df_dup_v6.drop(
    df_dup_v6[df_dup_v6['songName'] == 'Let Me Leave You'].index
).reset_index(drop=True)

In [31]:
df_dup_v6.loc[df_dup_v6['songName'] == 'Let Me Leave You', VIEW_COLS]

,songName,artists,api_channel_title,api_view_count,api_published_at,video_id,url


> 유통사 채널 필터 이후에도 같은 곡(`MOVIE`)이 다른 채널에 중복 업로드된 상태로 남아있어, 하나만 남기고 제거합니다.

In [32]:
df_dup_v6 = df_dup_v6.drop(
    df_dup_v6[df_dup_v6['url'] == 'https://www.youtube.com/watch?v=-Rv_K7qy4Ok'].index
).reset_index(drop=True)

In [33]:
df_dup_v6.loc[df_dup_v6['songName'] == 'MOVIE', VIEW_COLS]

,songName,artists,api_channel_title,api_view_count,api_published_at,video_id,url
243,MOVIE,2921|박제업||0,PARK JEUP,14493.0,2025-07-25 09:00:02+00:00,0MWJo1C24ww,https://www.youtube.com/watch?v=0MWJo1C24ww


> 중복 확인 과정에서 발견된 저조회수 영상(`영화 한편 본 것 같아`, 조회수 1,851회)으로, 두 업로드 모두 데이터셋에서 제외합니다.

In [34]:
df_dup_v6 = df_dup_v6.drop(
    df_dup_v6[df_dup_v6['url'] == 'https://www.youtube.com/watch?v=i2FkyWgBMA4'].index
).reset_index(drop=True)

In [35]:
df_dup_v6.loc[df_dup_v6['songName'] == '영화 한편 본 것 같아', VIEW_COLS]

,songName,artists,api_channel_title,api_view_count,api_published_at,video_id,url
294,영화 한편 본 것 같아,1561|송하예||0,에잇 8recordz x studio8,1851.0,2025-09-17 09:00:23+00:00,Fag8mxHphxA,https://www.youtube.com/watch?v=Fag8mxHphxA


In [36]:
df_dup_v6 = df_dup_v6.drop(
    df_dup_v6[df_dup_v6['url'] == 'https://www.youtube.com/watch?v=Fag8mxHphxA'].index
).reset_index(drop=True)

In [37]:
df_dup_v6.loc[df_dup_v6['songName'] == '영화 한편 본 것 같아', VIEW_COLS]

,songName,artists,api_channel_title,api_view_count,api_published_at,video_id,url


## 5. 중복 제거가 끝난 데이터 병합
<small>채널 정리까지 끝난 중복 후보(`df_dup_v6`)를 애초에 중복 없던 곡(`df_unique`)과 합쳐 최종본을 만들고, 병합 전에 검증합니다. 마지막에 기존 결과와 비교합니다.</small>

In [38]:
# 1. video_id 중복 확인
print(f"video_id 중복: {df_dup_v6['video_id'].duplicated().sum()} 개")

# 2. 채널명 2개 이상인 곡 확인
result_v6 = df_dup_v6.groupby(['songName', 'artists'])['api_channel_title'].nunique()
print(f"채널명 2개 이상인 곡: {len(result_v6[result_v6 >= 2])} 개")

print(f"df_dup_v6: {len(df_dup_v6)} rows | df_unique: {len(df_unique)} rows")

video_id 중복: 0 개
채널명 2개 이상인 곡: 0 개
df_dup_v6: 410 rows | df_unique: 1633 rows


In [39]:
# 겹치는 곡 확인
overlap = df_unique.set_index(['songName', 'artists']).index.isin(
    df_dup_v6.set_index(['songName', 'artists']).index
)
print(f"겹치는 곡: {overlap.sum()} 개")

겹치는 곡: 0 개


In [40]:
df_final = pd.concat([df_unique, df_dup_v6]).reset_index(drop=True)
print(f"df_unique: {len(df_unique)} rows")
print(f"df_dup_v6: {len(df_dup_v6)} rows")
print(f"최종: {len(df_final)} rows")

df_unique: 1633 rows
df_dup_v6: 410 rows
최종: 2043 rows


> 채널 기준으로는 걸러지지 않았지만, 같은 공식 채널이 올린 서로 다른 영상(뮤직비디오와 퍼포먼스 영상 등)이 한 곡에 여러 개 남는 경우가 있습니다. 유통사 영상만 제거하고 나머지는 모두 유지하는 것이 이 노트북의 기준이라 그대로 둡니다.

In [41]:
# 곡당 영상이 2개 이상 남은 곡
multi_video = df_final.groupby(['songName', 'artists']).size()
print(f"곡당 영상이 2개 이상 남은 곡: {(multi_video > 1).sum()}곡")

곡당 영상이 2개 이상 남은 곡: 5곡


> **기존 결과와 비교:** 출력 파일이 이미 있으면 새 결과와 비교해 일치 여부를 출력합니다(행 순서와 값까지 비교). 저장은 `SAVE_OUTPUT = True`일 때(또는 기존 파일이 없을 때)만 합니다.

In [42]:
# 기존 결과와 비교하고, SAVE_OUTPUT = True일 때만 저장
output_exists = OUTPUT_FILE.exists()

if output_exists:
    previous = pd.read_csv(OUTPUT_FILE, encoding='utf-8-sig')
    same_ids = set(previous['video_id']) == set(df_final['video_id'])
    try:
        pd.testing.assert_frame_equal(previous, df_final, check_dtype=False)   # 행 순서와 값까지 비교 (부동소수점 오차는 허용)
        verdict = "행 순서와 모든 값이 동일"
    except AssertionError:
        verdict = "video_id는 같지만 행 순서나 값에 차이가 있음" if same_ids else "video_id가 다름"
    print(f"🔍 기존 출력 파일과 비교: {verdict} (기존 {len(previous):,}행 / 이번 {len(df_final):,}행)")
else:
    print("🔍 기존 출력 파일이 없어 비교를 건너뜁니다.")

if SAVE_OUTPUT or not output_exists:
    df_final.to_csv(OUTPUT_FILE, index=False)      # 기존 결과와 같은 형식(UTF-8, BOM 없음)으로 저장
    reason = "SAVE_OUTPUT = True" if SAVE_OUTPUT else "기존 출력 파일이 없어서"
    print(f"📁 저장 완료 ({reason}): {OUTPUT_FILE}")
else:
    print(f"📁 저장하지 않음: 기존 {OUTPUT_FILE.name} 유지 (저장하려면 SAVE_OUTPUT = True)")

🔍 기존 출력 파일과 비교: 행 순서와 모든 값이 동일 (기존 2,043행 / 이번 2,043행)
📁 저장하지 않음: 기존 data(drop_duplicated).csv 유지 (저장하려면 SAVE_OUTPUT = True)
